In [4]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score


In [8]:
train = pd.read_csv('/Users/karthikx06/Documents/IDX_Exchange/data/notebooks/data/processed/train_clean.csv')
test = pd.read_csv('/Users/karthikx06/Documents/IDX_Exchange/data/notebooks/data/processed/test_clean.csv')

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")

Train shape: (59316, 68)
Test shape: (12000, 68)


In [9]:
baseline_features = [
    'LivingArea',
    'BedroomsTotal',
    'BathroomsTotalInteger',
    'LotSizeSquareFeet',
    'YearBuilt',
    'Latitude',
    'Longitude'
]

target = 'log_price'  # your log1p(ClosePrice) from Week 3

# Sanity check: no NaNs in features or target before fitting
print("Train NaNs:\n", train[baseline_features + [target]].isna().sum())
print("\nTest NaNs:\n", test[baseline_features + [target]].isna().sum())

Train NaNs:
 LivingArea               0
BedroomsTotal            0
BathroomsTotalInteger    0
LotSizeSquareFeet        0
YearBuilt                0
Latitude                 0
Longitude                0
log_price                0
dtype: int64

Test NaNs:
 LivingArea               0
BedroomsTotal            0
BathroomsTotalInteger    0
LotSizeSquareFeet        0
YearBuilt                0
Latitude                 0
Longitude                0
log_price                0
dtype: int64


In [10]:
X_train = train[baseline_features]
y_train = train[target]

X_test = test[baseline_features]
y_test_log = test[target]

# Keep the actual dollar ClosePrice for test set — needed for MAPE/MdAPE
y_test_actual = test['ClosePrice']


In [11]:
model = LinearRegression()
model.fit(X_train, y_train)


,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](7,)","[ 0. ,-0.02, 0.2 ,...,-0.01,-0.04,-0.02]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](7,)","['LivingArea','BedroomsTotal','BathroomsTotalInteger',...,'YearBuilt', 'Latitude','Longitude']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,25.8
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,7
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int,7


In [12]:
y_pred_log = model.predict(X_test)
y_pred_actual = np.expm1(y_pred_log)  # inverse of log1p from Week 3

In [13]:
def mape(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

def mdape(y_true, y_pred):
    return np.median(np.abs((y_true - y_pred) / y_true)) * 100

# R² — reporting both scales for transparency
r2_log = r2_score(y_test_log, y_pred_log)
r2_dollar = r2_score(y_test_actual, y_pred_actual)

mape_score = mape(y_test_actual, y_pred_actual)
mdape_score = mdape(y_test_actual, y_pred_actual)

print("=== Baseline Linear Regression — Results ===")
print(f"R² (log scale):     {r2_log:.4f}")
print(f"R² (dollar scale):  {r2_dollar:.4f}")
print(f"MAPE:               {mape_score:.2f}%")
print(f"MdAPE:              {mdape_score:.2f}%")


=== Baseline Linear Regression — Results ===
R² (log scale):     0.4529
R² (dollar scale):  -4646.9526
MAPE:               47.75%
MdAPE:              29.76%


In [16]:
pd.set_option('display.width', 150)
pd.set_option('display.max_colwidth', 30)

results_df = pd.DataFrame([results_log])
results_df.to_csv('model_results_log.csv', index=False)

# Round for readability
display_df = results_df.copy()
display_df['r2_log_scale'] = display_df['r2_log_scale'].round(4)
display_df['r2_dollar_scale'] = display_df['r2_dollar_scale'].round(2)
display_df['mape'] = display_df['mape'].round(2)
display_df['mdape'] = display_df['mdape'].round(2)

print(display_df[['model_name', 'r2_log_scale', 'r2_dollar_scale', 'mape', 'mdape']])

            model_name  r2_log_scale  r2_dollar_scale   mape  mdape
0  Linear Regression A        0.4529         -4646.95  47.75  29.76


In [17]:
# Build a results dataframe: actual price, predicted price, error, absolute error
diagnostics = pd.DataFrame({
    'actual': y_test_actual.values,
    'predicted': y_pred_actual,
})
diagnostics['abs_error'] = np.abs(diagnostics['actual'] - diagnostics['predicted'])
diagnostics['pct_error'] = diagnostics['abs_error'] / diagnostics['actual'] * 100

# Sort by worst absolute dollar error
# Format floats as plain numbers with comma separators, no scientific notation
pd.set_option('display.float_format', lambda x: f'{x:,.0f}')

worst_predictions = diagnostics.sort_values('abs_error', ascending=False).head(15)
print(worst_predictions)

          actual      predicted      abs_error  pct_error
11998 21,500,000 12,470,107,665 12,448,607,665     57,901
11983 10,380,000    663,424,858    653,044,858      6,291
10176 97,972,500        961,862     97,010,638         99
559   11,000,000     88,251,639     77,251,639        702
9485   8,400,000     80,080,254     71,680,254        853
893   24,000,000     89,782,845     65,782,845        274
8113  48,720,000        778,370     47,941,630         98
10428 17,000,000     52,038,630     35,038,630        206
46    35,000,000      5,364,911     29,635,089         85
1725  31,000,000     59,538,625     28,538,625         92
11896 17,000,000     43,440,728     26,440,728        156
2819  10,100,000     34,464,327     24,364,327        241
390   28,000,000      4,310,603     23,689,397         85
10102  4,850,000     26,569,951     21,719,951        448
189   18,277,380     36,879,845     18,602,465        102
